# セミナーII 第2回 実践：画像と文章の類似度で分類・検索する（CLIP）

**CLIP** は、画像と文章を、同じ512次元のベクトル空間に埋め込むモデルです。画像のベクトルと文章のベクトルのコサイン類似度を計算すれば、

- 画像に一番近い文章を候補から選ぶ → 好きな言葉をクラスにした分類（**ゼロショット分類**）
- 文章に一番近い画像を探す → 文章による画像の検索

ができます。画像の側には、講義で見た ViT が使われています。第1回では37品種の分類を学習しましたが、CLIP はこれを学習なしで解けます（5-3）。

**最初に**

1. メニューの「ファイル」→「ドライブにコピーを保存」で、自分のコピーを作ります。
2. 「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選びます。
3. 上のセルから順に実行します（セルの左の ▶、または **Shift + Enter**）。

ログインや Google ドライブへのアクセスの許可は必要ありません。全体で約35分（うち計算 約3分）です。

- セルの右に入力欄やプルダウンが出るセルでは、そこで値を選んでから実行します（コードを書き換えても同じです）。
- 図を描く関数のセルは、コードを畳んであります。実行するだけでかまいません。
- このモデルは英語の文章で学習しています。ラベルや検索の文章は英語で書いてください。
- しばらく操作しないと接続が切れます。`NameError` が出たときは、1. から順に実行し直してください。

## 0. このノートブックの流れ

目安：約1分

| 節 | やること |
| --- | --- |
| 1. 準備 | ライブラリとモデルを読み込む |
| 2. 手法を知る | 第1回の分類との違い、CLIP の仕組み、学習のしかた（対照学習） |
| 3. 例の画像で試す | ゼロショット分類を、途中の値を見ながら1枚ずつ追う |
| 4. 自分の画像で試す | 自分の写真を、自分で決めたラベルで分類する |
| 5. やってみよう | ラベルの文章を変える、候補にない答え、第1回の37品種を解く |
| 6. 活用 | 文章で画像を検索する、似た画像を検索する |
| 7. まとめ | 分かったことと、苦手なこと |

## 1. 準備

目安：約3分（うち計算 約1分）

`device: cuda` と GPU の名前が表示されれば準備完了です。`device: cpu` と表示されたら、「ランタイム」→「ランタイムのタイプを変更」で GPU を選んで、もう一度実行してください。

In [ ]:
import importlib.util
import io
import subprocess
import sys
import time

# Colab に入っていないライブラリがあれば入れる
for package in ["transformers", "datasets", "pooch"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from PIL import Image, ImageOps
from skimage import data
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32  # GPU では半分の精度で速く計算する
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

CLIP（ViT-B/32、約1億5000万パラメータ）を読み込み、画像と文章をベクトルにする関数を定義します。初回は重み（約600MB）をダウンロードします。`HF_TOKEN` に関する警告が表示されても無視してかまいません。

- モデル：[laion/CLIP-ViT-B-32-laion2B-s34B-b79K](https://huggingface.co/laion/CLIP-ViT-B-32-laion2B-s34B-b79K)（MIT）。LAION が公開している、CLIP と同じ仕組みのモデル（OpenCLIP）。モデルカードでは、研究のための利用を想定し、英語以外での利用や、製品への組み込みは対象外としています
- 論文：A. Radford et al., "Learning Transferable Visual Models From Natural Language Supervision," ICML 2021（[arXiv:2103.00020](https://arxiv.org/abs/2103.00020)）

In [ ]:
CLIP_ID = "laion/CLIP-ViT-B-32-laion2B-s34B-b79K"
processor = CLIPProcessor.from_pretrained(CLIP_ID)
model = CLIPModel.from_pretrained(CLIP_ID, dtype=dtype).to(device).eval()
SCALE = model.logit_scale.exp().item()  # コサイン類似度にかける倍率（学習で決まった値。2-2）


def _vectors(output):
    # transformers の版によって、ベクトルそのものか、ベクトルを含む出力が返るので、そろえる
    return output if torch.is_tensor(output) else output.pooler_output


def encode_images(images, batch=128):
    # 画像のリスト → 長さを1に正規化したベクトル（枚数 × 512）
    out = []
    for i in range(0, len(images), batch):
        pixel_values = processor(images=[im.convert("RGB") for im in images[i:i + batch]],
                                 return_tensors="pt")["pixel_values"].to(device, dtype)
        with torch.no_grad():
            out.append(F.normalize(_vectors(model.get_image_features(pixel_values=pixel_values)).float(), dim=-1).cpu())
    return torch.cat(out)


def encode_texts(texts):
    # 文章のリスト → 長さを1に正規化したベクトル（文の数 × 512）
    inputs = processor(text=list(texts), return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        vectors = model.get_text_features(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
    return F.normalize(_vectors(vectors).float(), dim=-1).cpu()


def classify(image, labels):
    # ゼロショット分類：コサイン類似度 × SCALE を softmax で確率にする（3. で1段ずつ確かめる）
    similarity = encode_images([image]) @ encode_texts(labels).T  # (1, ラベルの数)
    return (similarity[0] * SCALE).softmax(dim=0)


print(f"パラメータの数: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M"
      f"（画像の側 {sum(p.numel() for p in model.vision_model.parameters()) / 1e6:.0f}M、"
      f"文章の側 {sum(p.numel() for p in model.text_model.parameters()) / 1e6:.0f}M）")

図を描く関数と、例の画像を用意します。

In [ ]:
# @title 図を描く関数と、例の画像（実行するだけでよい） { display-mode: "form" }
PETS_ID = "timm/oxford-iiit-pet"  # 第1回と同じデータセット
PETS_REVISION = "089695c834a7deb60505b7cc506672db1c31a6aa"
_pets = {}

images = {  # scikit-image に入っている写真
    "cat": Image.fromarray(data.chelsea()),                      # 撮影 Stefan van der Walt（CC0）
    "astronaut": Image.fromarray(data.astronaut()),              # NASA（パブリックドメイン）
    "coffee": Image.fromarray(data.coffee()),                    # 撮影 Rachel Michetti（CC0）
    "rocket": Image.fromarray(data.rocket()),                    # SpaceX（パブリックドメイン）
    "motorcycle": Image.fromarray(data.stereo_motorcycle()[0]),  # Middlebury 2014 ステレオデータセット
    "coins": Image.fromarray(data.coins()).convert("RGB"),       # Brooklyn Museum（著作権の制限は知られていない）
}


def split_labels(text):
    return [t.strip() for t in text.split(",") if t.strip()]


def show_probs(image, labels, probs, title=""):
    # 画像と、各ラベルの確率を並べる
    order = probs.argsort(descending=True)
    fig, axes = plt.subplots(1, 2, figsize=(11, max(2.6, 0.45 * len(labels) + 1)), width_ratios=[1, 1.5])
    axes[0].imshow(image)
    axes[0].set_title(title)
    axes[0].axis("off")
    axes[1].barh([labels[i] for i in order][::-1], probs[order].numpy()[::-1])
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel("probability")
    for k, i in enumerate(order.tolist()[::-1]):
        axes[1].text(min(probs[i].item() + 0.02, 0.85), k, f"{probs[i]:.2f}", va="center")
    plt.tight_layout()
    plt.show()


def show_questions(questions, columns=3):
    # (画像の名前, [ラベル]) のリストを、小さな図にまとめて並べる
    rows = (len(questions) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns * 2, figsize=(6.4 * columns, 2.6 * rows), squeeze=False,
                             width_ratios=[1, 1.6] * columns)
    for ax in axes.flat:
        ax.axis("off")
    for k, (name, labels) in enumerate(questions):
        ax_image, ax_bar = axes[k // columns, (k % columns) * 2], axes[k // columns, (k % columns) * 2 + 1]
        probs = classify(images[name], labels)
        ax_image.imshow(ImageOps.fit(images[name], (224, 224)))
        ax_image.set_title(name, fontsize=10)
        ax_bar.axis("on")
        ax_bar.barh(labels[::-1], probs.numpy()[::-1])
        ax_bar.set_xlim(0, 1)
        ax_bar.tick_params(labelsize=9)
        for j, p in enumerate(probs.numpy()[::-1]):
            ax_bar.text(min(p + 0.03, 0.8), j, f"{p:.2f}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()


def show_similarity(names, texts, similarity):
    # 画像（行）と文章（列）のコサイン類似度の表
    fig, ax = plt.subplots(figsize=(1.6 * len(texts) + 2, 1.3 * len(names)))
    ax.imshow(similarity.numpy(), cmap="viridis", extent=(0, len(texts), len(names), 0))
    for i in range(len(names)):
        for j in range(len(texts)):
            ax.text(j + 0.5, i + 0.5, f"{similarity[i, j]:.2f}", ha="center", va="center", color="white", fontsize=11)
        ax.imshow(ImageOps.fit(images[names[i]], (224, 224)), extent=(-1.1, -0.1, i + 0.95, i + 0.05))
    ax.set_xlim(-1.2, len(texts))  # 画像を描くと表示の範囲が変わるので、最後に決め直す
    ax.set_ylim(len(names), 0)
    ax.set_xticks(np.arange(len(texts)) + 0.5, texts, rotation=20, ha="right")
    ax.set_yticks([])
    ax.set_title("cosine similarity (image x text)")
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    plt.show()


def load_pets():
    # 第1回のデータセットの test（3669枚）と、その画像のベクトル。1回だけ計算して使い回す
    if not _pets:
        start = time.time()
        test = load_dataset(PETS_ID, revision=PETS_REVISION, split="test")
        _pets["data"] = test
        _pets["names"] = [n.replace("_", " ") for n in test.features["label"].names]
        _pets["labels"] = torch.tensor(test["label"])
        _pets["features"] = encode_images(test["image"])
        print(f"test の {len(test)} 枚をベクトルにしました（{time.time() - start:.0f} 秒）")
    return _pets


def show_gallery(indices, captions, colors=None, title="", columns=8):
    # データセットの画像を並べる
    pets = load_pets()
    rows = (len(indices) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(2.1 * columns, 2.5 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for k, (ax, i) in enumerate(zip(axes.flat, indices)):
        ax.imshow(ImageOps.fit(pets["data"][int(i)]["image"].convert("RGB"), (224, 224)))
        ax.set_title(captions[k], fontsize=8, color="black" if colors is None else colors[k])
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def fit_size(image, max_side=1024):
    # 大きすぎる写真は縮小する
    image = image.convert("RGB")
    scale = max_side / max(image.size)
    if scale < 1:
        image = image.resize((round(image.width * scale), round(image.height * scale)), Image.LANCZOS)
    return image


print("例の画像:", list(images))

## 2. 手法を知る

目安：約8分

### 2-1. 第1回の分類との違い

第1回のモデルは、最後の**分類ヘッド**（全結合層）が、512次元の特徴ベクトルから37品種のスコアを出していました。分類ヘッドの重み（37×512）は学習で決めるので、クラスを変えるには、正解を付けた画像を集めて学習し直す必要があります。

CLIP は、画像と文章の両方を、**同じ512次元の空間**のベクトルにします。意味が対応する画像と文章のベクトルは、コサイン類似度が大きくなるように学習してあります。分類するときは、ラベルを文章（"a photo of a cat" など）にしてベクトルにし、画像のベクトルとのコサイン類似度が一番大きいものを答えにします。ラベルは好きな文章にできるので、学習し直さずに、新しいクラスで分類できます（**ゼロショット分類**）。

### 2-2. CLIP の仕組み

| 部品 | 処理 |
| --- | --- |
| 画像エンコーダ（ViT-B/32） | 画像を 224×224 にそろえ、32×32 画素のパッチ（7×7 = 49個）をトークンにして Transformer に入れ、512次元のベクトルにする |
| 文章エンコーダ（Transformer） | 文章を単語（の断片）のトークンに分けて Transformer に入れ、512次元のベクトルにする |
| 類似度 | 両方のベクトルを長さ1に正規化し、内積（＝コサイン類似度）を計算する |

ゼロショット分類では、画像と各ラベルのコサイン類似度に倍率（このモデルでは100）をかけ、softmax で確率にします。これは、第1回の分類ヘッドと同じ「重み × 特徴ベクトル → softmax」の形で、**重みの各行を、学習で決める代わりに、ラベルの文章のベクトルにしたもの**です（バイアスはなし。論文でもこの見方を説明しています）。

下のセルで、5枚の画像と5つの文章をベクトルにして、コサイン類似度の表を作ります。

In [ ]:
names = ["cat", "astronaut", "coffee", "rocket", "motorcycle"]
texts = ["a photo of a cat", "a photo of an astronaut", "a photo of a cup of coffee",
         "a photo of a rocket launch", "a photo of a motorcycle"]

image_vectors = encode_images([images[n] for n in names])  # (5, 512)
text_vectors = encode_texts(texts)                          # (5, 512)
print("画像のベクトル:", tuple(image_vectors.shape), " 長さ:", image_vectors.norm(dim=1).numpy().round(3))
print("文章のベクトル:", tuple(text_vectors.shape), " 長さ:", text_vectors.norm(dim=1).numpy().round(3))

similarity = image_vectors @ text_vectors.T  # 長さ1どうしの内積 = コサイン類似度。(5, 5)
show_similarity(names, texts, similarity)

各行で、対角線（正しい組）の値が一番大きくなっています。コサイン類似度の値そのものは 0.1〜0.3 程度で差も小さいですが、倍率100をかけてから softmax にするので、確率では大きな差になります（3. で確かめます）。

### 2-3. どうやって学習したか（対照学習）

CLIP は、Web から集めた**画像と、その説明文の組**で学習します。人が正解のラベルを付ける必要はありません。論文の CLIP は4億組で学習しました。このノートブックのモデルは、LAION-5B（58.5億組）の英語の部分（約20億組）で学習しています。

学習のしかたは、上の表で説明できます。

1. ミニバッチの N 組の画像と文章を、それぞれベクトルにする
2. N×N のコサイン類似度の表を作る
3. **対角線（正しい組）を大きく、それ以外（間違った組）を小さく**するように、両方のエンコーダの重みを更新する

3. の Loss は、表の各行で「この画像に合う文章は N 個のうちどれか」、各列で「この文章に合う画像はどれか」を当てる分類の**交差エントロピー**です。第1回と同じ Loss を、クラスの代わりにミニバッチの中の文章（画像）に対して使っています。論文の CLIP のミニバッチは 32,768 組でした。倍率（温度の逆数）も学習するパラメータで、100 を上限にしています。

## 3. 例の画像で試す

目安：約5分

猫の画像を、3つのラベルでゼロショット分類します。2-2 の計算を1段ずつ実行し、途中の値を表示します。

In [ ]:
image = images["cat"]
labels = ["a photo of a cat", "a photo of a dog", "a photo of a car"]

text_vectors = encode_texts(labels)            # ① ラベルの文章をベクトルに：(3, 512)。分類ヘッドの重みにあたる
image_vector = encode_images([image])          # ② 画像をベクトルに：(1, 512)
similarity = image_vector @ text_vectors.T     # ③ コサイン類似度：(1, 3)
probs = (similarity[0] * SCALE).softmax(dim=0)  # ④ 倍率をかけて softmax：確率（合計1）

print("① ラベルのベクトル:", tuple(text_vectors.shape), " ② 画像のベクトル:", tuple(image_vector.shape))
for label, s, p in zip(labels, similarity[0], probs):
    print(f"  {label:20s}  ③ コサイン類似度 {s:.3f}  →  × {SCALE:.0f} = {s * SCALE:5.1f}  →  ④ 確率 {p:.3f}")
show_probs(image, labels, probs, "cat")

コサイン類似度の差は 0.1〜0.2 程度ですが、100倍すると 10〜20 の差になり、softmax で確率がほぼ1と0に分かれます。

同じ8個のラベルで、6枚の画像を分類します（`classify` は ①〜④ をまとめた関数です）。どのラベルも、学習で決まっていたクラスではなく、ここで書いた文章です。

In [ ]:
LABELS = ["a photo of a cat", "a photo of a dog", "a photo of an astronaut", "a photo of a cup of coffee",
          "a photo of a rocket", "a photo of a motorcycle", "a photo of old coins", "a photo of a car"]
show_questions([(name, LABELS) for name in images], columns=2)

## 4. 自分の画像で試す

目安：約5分（画像を用意する時間は除く）

`USE_UPLOAD` にチェックを入れて実行すると、アップロードのボタンが表示されます（Colab のみ）。アップロードした画像は、ファイル名（拡張子を除く）の名前で、5. と 6. でも使えます。`LABELS` に、カンマで区切って英語のラベルを書きます。

In [ ]:
USE_UPLOAD = False  # @param {type:"boolean"}
LABELS = "a photo of a cat, a photo of a dog, a photo of a person, a photo of food, a photo of a building"  # @param {type:"string"}

if USE_UPLOAD:
    from google.colab import files

    for filename, content in files.upload().items():
        name = filename.rsplit(".", 1)[0]
        images[name] = fit_size(Image.open(io.BytesIO(content)))
        labels = split_labels(LABELS)
        show_probs(images[name], labels, classify(images[name], labels), name)
print("使える画像の名前:", list(images))

## 5. やってみよう

目安：約10分（うち計算 約1分）

### 5-1. ラベルの文章を変える

物の名前だけでなく、色、表情、時間帯、場所、写り方なども、ラベルの文章にできます。画像ごとに2〜3個のラベルを用意しました。**実行する前に、どれが選ばれるか予想してください。**

`QUESTIONS` のラベルを書き換えたり、行を足したりしてみましょう。コーヒーとお茶のように、確率が分かれる（モデルが迷う）例も探してみてください。

In [ ]:
QUESTIONS = [  # (画像の名前, [ラベル])
    ("cat", ["a cat with green eyes", "a cat with blue eyes"]),
    ("cat", ["a tabby cat", "a black cat", "a white cat"]),
    ("astronaut", ["a smiling person", "a crying person", "an angry person"]),
    ("coffee", ["a cup of coffee", "a cup of tea", "a glass of beer"]),
    ("rocket", ["a photo taken during the day", "a photo taken at night"]),
    ("motorcycle", ["a motorcycle in a garage", "a motorcycle on a road"]),
    ("coins", ["ancient coins", "modern coins"]),
    ("coins", ["a color photo", "a black and white photo"]),
]
show_questions(QUESTIONS)

### 5-2. 候補にない答えは出せない

softmax は、渡したラベルの中で確率を配分するだけです。宇宙飛行士の画像を「猫か犬か」で分類すると、どうなるでしょうか。第1回のモデルが、コーヒーカップの写真にも37品種のどれかで答えたのと同じです。

In [ ]:
show_questions([
    ("astronaut", ["a photo of a cat", "a photo of a dog"]),
    ("astronaut", ["a photo of a cat", "a photo of a dog", "a photo of a person"]),
], columns=2)

### 5-3. 第1回の37品種を、学習なしで解く

第1回の test（3669枚）を、37品種の名前をラベルにしてゼロショット分類します。第1回では、ResNet-18 を train の約2900枚で学習して、test の正解率が約0.83でした。

計算は 3. と同じで、まとめて行列で計算します。

- ラベルのベクトル：37品種 → (37, 512)。分類ヘッドの重みにあたる
- 画像のベクトル：3669枚 → (3669, 512)
- コサイン類似度：(3669, 512) × (512, 37) → (3669, 37)。各行で最大の列が答え

`TEMPLATE` はラベルの文章の形で、`{}` に品種名が入ります。初回は、データセットのダウンロードと、3669枚のベクトルの計算に1分ほどかかります。下には、ランダムに選んだ16枚の答えを表示します（緑：正解、赤：不正解）。

In [ ]:
TEMPLATE = "a photo of a {}."  # @param ["a photo of a {}.", "a photo of a {}, a type of pet."] {allow-input: true}

pets = load_pets()
class_vectors = encode_texts([TEMPLATE.format(n) for n in pets["names"]])  # (37, 512)
similarity = pets["features"] @ class_vectors.T                           # (3669, 37)
preds = similarity.argmax(dim=1)
accuracy = (preds == pets["labels"]).float().mean().item()
bombay = pets["names"].index("bombay")
print(f"ラベルの形: {TEMPLATE!r}   類似度の行列: {tuple(similarity.shape)}")
print(f"正解率: {accuracy:.3f}（第1回の学習したモデルは約0.83）")
print(f"bombay（黒猫の品種）の正解率: {(preds[pets['labels'] == bombay] == bombay).float().mean():.2f}")

picked = torch.randperm(len(preds), generator=torch.Generator().manual_seed(0))[:16]
captions = [f"pred: {pets['names'][preds[i]]}\ntrue: {pets['names'][pets['labels'][i]]}" for i in picked]
colors = ["tab:green" if preds[i] == pets["labels"][i] else "tab:red" for i in picked]
show_gallery(picked, captions, colors, f"zero-shot (accuracy {accuracy:.3f})")

`"a photo of a {}."` では、bombay の正解率がほぼ0です。Bombay はインドの都市（ムンバイの旧名）の名前でもあるので、"a photo of a bombay" が猫の写真のことだと伝わりにくいのです。`TEMPLATE` を `"a photo of a {}, a type of pet."` に変えて、bombay と全体の正解率がどう変わるか確かめましょう（論文でも、Oxford-IIIT Pet ではこの書き方がよかったと報告されています）。ラベルの文章を変えることは、分類ヘッドの重みを変えることにあたります。

**比べるときの注意**：CLIP は約20億組という、第1回とはけた違いの量のデータで学習しています。犬や猫の写真や品種名も大量に含まれていたはずで、この test の写真そのものが含まれていた可能性も否定できません。一方、第1回の方法なら、CLIP が知らない物（研究で扱う特殊な画像など）でも、正解を付けた画像を集めれば学習できます。

## 6. 活用：ベクトルで検索する

目安：約6分

5-3 で計算した test の3669枚の画像のベクトル (3669, 512) を、検索の対象にします（5-3 を飛ばした人は、初回に1分ほどかかります）。

### 6-1. 文章で画像を検索する

検索の文章をベクトル (1, 512) にし、3669枚のベクトルとのコサイン類似度 (3669,) を計算して、大きい順に16枚並べます。画像には品種名しか付いていません。「雪の中」「衣装を着た」のような情報は、どこにも書かれていないことに注目してください。

In [ ]:
QUERY = "a dog playing in the snow"  # @param ["a dog playing in the snow", "a cat sleeping on a bed", "a dog wearing a costume", "a black and white photo of a dog", "a puppy", "two cats"] {allow-input: true}

pets = load_pets()
scores = (pets["features"] @ encode_texts([QUERY]).T)[:, 0]  # (3669,)
top = scores.argsort(descending=True)[:16]
show_gallery(top, [f"{pets['names'][pets['labels'][i]]}\n{scores[i]:.3f}" for i in top], title=f'"{QUERY}"')

### 6-2. 似た画像を検索する

検索に使うベクトルを、文章ではなく画像のベクトルにすると、似た画像の検索になります。`IMAGE` に画像の名前を選びます（自分の画像は名前を入力します）。

In [ ]:
IMAGE = "cat"  # @param ["cat", "astronaut", "coffee", "motorcycle"] {allow-input: true}

pets = load_pets()
scores = (pets["features"] @ encode_images([images[IMAGE]]).T)[:, 0]  # (3669,)
top = scores.argsort(descending=True)[:16]
plt.figure(figsize=(3, 3))
plt.imshow(images[IMAGE])
plt.title(f"query: {IMAGE}")
plt.axis("off")
plt.show()
show_gallery(top, [f"{pets['names'][pets['labels'][i]]}\n{scores[i]:.3f}" for i in top], title="most similar images")

## 7. まとめ

目安：約2分

| 分かったこと | 節 |
| --- | --- |
| CLIP は、画像と文章を同じ512次元の空間のベクトルにし、コサイン類似度で比べる | 2-2 |
| ゼロショット分類は、第1回の分類ヘッドの重みを、ラベルの文章のベクトルで置き換えたもの | 2-2、3. |
| 学習は、N×N の類似度の表の対角を大きくする対照学習。Loss は交差エントロピー | 2-3 |
| ラベルの文章を変えるだけで、色や様子なども分類できる。答えは渡したラベルのどれか | 5-1、5-2 |
| 第1回の37品種を、学習なしで高い正解率で解ける。ラベルの書き方で結果が変わる | 5-3 |
| コサイン類似度の大きい順に並べるだけで、文章や画像で検索できる | 6. |

**苦手なこと**：数を数える（論文でも報告されています）、位置関係（「左に犬、右に猫」）、否定（「猫ではない」）、専門的な画像（顕微鏡、衛星画像など）。自分の画像で確かめてみましょう（例：犬が2匹の写真で、"one dog" と "two dogs" を比べる）。